In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="hRTUI0MqBME6xajesmw7")
project = rf.workspace("cam-ctadw").project("makinegormesi-6bypv")
version = project.version(2)
dataset = version.download("coco")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to MakineGormesi-2 in coco:: 100%|██████████| 1551/1551 [00:00<00:00, 5568.09it/s]


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pycocotools.coco import COCO
from torchvision import transforms

CLASSES = ['cam', 'kagit', 'metal', 'pil', 'plastik'] # kendi sınıfların
NUM_CLASSES = len(CLASSES) + 1  # + background
IMG_SIZE = 512
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/content/MakineGormesi-2"  # COCO JSON dosyalarının olduğu klasör

class COCODataset(Dataset):
    def __init__(self, img_dir, json_file, transform=None):
        self.img_dir = img_dir
        self.coco = COCO(json_file)
        self.ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        path = os.path.join(self.img_dir, img_info['file_name'])
        image = Image.open(path).convert("RGB")

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes = []
        labels = []

        for ann in anns:
            xmin, ymin, w, h = ann['bbox']
            xmax = xmin + w
            ymax = ymin + h
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(ann['category_id'])  # COCO category_id 1'den başlıyor

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }

        if self.transform:
            image = self.transform(image)

        return image, target

# Transform ve DataLoader
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_dataset = COCODataset(
    img_dir=os.path.join(DATA_DIR, "train"),
    json_file=os.path.join(DATA_DIR, "train/_annotations.coco.json"),
    transform=transform
)

val_dataset = COCODataset(
    img_dir=os.path.join(DATA_DIR, "valid"),
    json_file=os.path.join(DATA_DIR, "valid/_annotations.coco.json"),
    transform=transform
)

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


In [3]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

# --- 1. PARAMETRELER (YOLO İLE EŞİTLENDİ) ---
DATA_DIR = "/content/MakineGormesi-2"
PROJECT_NAME = "atik_faster_rcnn"
SAVE_DIR = f"runs/detect/{PROJECT_NAME}"

EPOCHS = 50             # YOLO: epochs=50
BATCH_SIZE = 16         # YOLO: batch=16 (HATA ALIRSAN 4 YAP!)
IMG_SIZE = 512          # YOLO: imgsz=512
NUM_WORKERS = 2         # Colab için genelde 2 idealdir
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Klasör oluştur
os.makedirs(SAVE_DIR, exist_ok=True)

# --- 2. MODEL TANIMLAMA (FASTER R-CNN) ---
def get_faster_rcnn_model(num_classes):
    # ResNet50 backbone kullanan model
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

    # Sınıflandırıcı kafasını değiştirme
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# Model başlatma (NUM_CLASSES önceki kodlarında tanımlıydı)
# Eğer tanımlı değilse manuel yaz: num_classes = 6 (5 sınıf + 1 arka plan)
try:
    model = get_faster_rcnn_model(NUM_CLASSES)
except NameError:
    # Güvenlik önlemi: NUM_CLASSES yoksa varsayılan 6 alıyorum
    print("⚠️ NUM_CLASSES bulunamadı, varsayılan olarak 6 (5 sınıf + 1 background) kullanılıyor.")
    model = get_faster_rcnn_model(6)

model.to(DEVICE)

# --- 3. VERİ YÜKLEME (TRANSFORM GÜNCELLEME) ---
# YOLO imgsz=512 yaptığı için biz de 512 yapıyoruz
train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((IMG_SIZE, IMG_SIZE)),
    torchvision.transforms.ToTensor(),
])

# Dataset Sınıfını Tekrar Çağırıyoruz (Adı COCODataset idi)
# train_dataset değişkenini güncellememiz lazım ki yeni parametreleri alsın
try:
    train_dataset = COCODataset(
        img_dir=os.path.join(DATA_DIR, "train"),
        json_file=os.path.join(DATA_DIR, "train/_annotations.coco.json"),
        transform=train_transform
    )
    # Loader'ı güncelle
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        collate_fn=collate_fn # Önceki koddan gelen fonksiyon
    )
    print(f"✅ Veri Seti Yüklendi: {len(train_dataset)} resim.")
except NameError:
    print("❌ HATA: 'COCODataset' veya 'collate_fn' tanımlı değil. Lütfen veri seti sınıfını tanımladığın hücreyi çalıştır.")

# --- 4. OPTIMIZER VE SCHEDULER ---
params = [p for p in model.parameters() if p.requires_grad]
# SGD genelde R-CNN için daha stabil sonuç verir
optimizer = optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
# Öğrenme oranını her epoch'ta biraz azaltır (YOLO benzeri davranış)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# --- 5. EĞİTİM DÖNGÜSÜ ---
print(f"\n🚀 Faster R-CNN Eğitimi Başlıyor...")
print(f"⚙️  Ayarlar: Epochs={EPOCHS}, Batch={BATCH_SIZE}, ImgSize={IMG_SIZE}")
print(f"📂 Kayıt Yeri: {SAVE_DIR}\n")

best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, targets in loop:
        try:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
            loop.set_postfix(loss=losses.item())

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("\n\n❌ HATA: GPU Belleği Yetmedi (Out of Memory)!")
                print("⚠️ Lütfen 'BATCH_SIZE' değerini 16'dan 4'e veya 2'ye düşürüp tekrar deneyin.")
                raise e
            else:
                raise e

    # Epoch sonu
    lr_scheduler.step()
    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Bitti | Avg Loss: {avg_loss:.4f}")

    # En iyi modeli kaydet
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best.pth"))

    # Son modeli kaydet
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, "last.pth"))

print(f"\n✅ Eğitim Tamamlandı! Modeller şurada: {SAVE_DIR}")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:02<00:00, 57.5MB/s]


loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
✅ Veri Seti Yüklendi: 1234 resim.

🚀 Faster R-CNN Eğitimi Başlıyor...
⚙️  Ayarlar: Epochs=50, Batch=16, ImgSize=512
📂 Kayıt Yeri: runs/detect/atik_faster_rcnn



Epoch 1/50: 100%|██████████| 78/78 [03:39<00:00,  2.82s/it, loss=0.184]


Epoch 1 Bitti | Avg Loss: 0.3161


Epoch 2/50: 100%|██████████| 78/78 [03:42<00:00,  2.85s/it, loss=0.21]


... (93 outputs left out)

Epoch 49/50: 100%|██████████| 78/78 [03:41<00:00,  2.84s/it, loss=0.0286]


Epoch 49 Bitti | Avg Loss: 0.0456


Epoch 50/50: 100%|██████████| 78/78 [03:42<00:00,  2.85s/it, loss=0.101]


Epoch 50 Bitti | Avg Loss: 0.0467

✅ Eğitim Tamamlandı! Modeller şurada: runs/detect/atik_faster_rcnn


In [4]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 22.5 MB/s eta 0:00:00


In [5]:
import torch
import time
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm import tqdm

# --- 1. AYARLAR ---
PROJECT_NAME = "atik_faster_rcnn"
SAVE_DIR = f"runs/detect/{PROJECT_NAME}"
MODEL_PATH = f"{SAVE_DIR}/best.pth"  # En iyi modeli yükleyeceğiz
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# --- 2. MODELİ YÜKLEME ---
print(f"📂 Model Yükleniyor: {MODEL_PATH}")

# Modeli tekrar tanımlıyoruz (Çünkü ağırlıkları yüklemek için boş modele ihtiyaç var)
try:
    # NUM_CLASSES global değişkeninden alıyoruz (eğitim kodunda tanımlıydı)
    model_eval = get_faster_rcnn_model(NUM_CLASSES)
except NameError:
    model_eval = get_faster_rcnn_model(6) # Varsayılan

# Ağırlıkları yükle
if os.path.exists(MODEL_PATH):
    model_eval.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("✅ Ağırlıklar başarıyla yüklendi.")
else:
    print(f"⚠️ UYARI: '{MODEL_PATH}' bulunamadı. Lütfen eğitimin bittiğinden emin ol.")
    # Eğitim bitmediyse mevcut modeli kullan (sadece test için)
    model_eval = model

model_eval.to(DEVICE)
model_eval.eval()

# --- 3. METRİK VE HIZ TESTİ ---
# O uyarıyı (100 detections) almamak veya önemsememek için varsayılan ayarları kullanıyoruz.
# R-CNN modelleri bazen çok fazla kutu önerir, bu normaldir.
metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

print("📊 Faster R-CNN Karşılaştırma Testi Başlıyor...")

start_time = time.time()
total_images = 0

# Test loader'ın tanımlı olduğundan emin olalım
if 'test_loader' not in locals():
    # Eğer test_loader yoksa validation loader'ı test niyetine kullanalım
    print("⚠️ test_loader bulunamadı, val_loader kullanılıyor.")
    test_loader = val_loader

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Faster R-CNN Test"):
        images = [img.to(DEVICE) for img in images]

        # Ground Truth verilerini hazırla
        target_data = []
        for t in targets:
            target_data.append({
                "boxes": t["boxes"].to(DEVICE),
                "labels": t["labels"].to(DEVICE)
            })

        # Model Tahmini
        # Faster R-CNN eval modunda direk: [{'boxes':..., 'scores':..., 'labels':...}] döner
        preds = model_eval(images)

        # Metrik hesaplayıcıya ver (Scores threshold uygulayabiliriz ama mAP hesabı için ham veri daha iyidir)
        metric.update(preds, target_data)

        total_images += len(images)

# --- 4. SONUÇLARI HESAPLA ---
end_time = time.time()
inference_time = end_time - start_time
fps = total_images / inference_time

results = metric.compute()

# --- 5. KARŞILAŞTIRMA RAPORU ---
print("\n" + "="*45)
print(f"📊 FASTER R-CNN vs SSD/YOLO KARŞILAŞTIRMA")
print("="*45)
print(f"⚡ HIZ (Performans):")
print(f"   - Toplam Süre      : {inference_time:.2f} saniye")
print(f"   - İşlenen Resim    : {total_images}")
print(f"   - FPS (Kare/Saniye): {fps:.2f}")
print("-" * 45)
print(f"🎯 BAŞARI (Accuracy):")
print(f"   - mAP@50 (Genel Başarı)     : {results['map_50'].item():.4f}")
print(f"   - mAP@50-95 (COCO Standardı): {results['map'].item():.4f}")
print("-" * 45)

if 'map_per_class' in results:
    print("📦 Sınıf Bazlı mAP@50-95 Değerleri:")
    # CLASSES listesi tanımlı değilse generic isimler ver
    try:
        class_names = ['background'] + CLASSES
    except NameError:
        class_names = [f"Class {i}" for i in range(len(results['map_per_class']) + 1)]

    for i, score in enumerate(results['map_per_class']):
        # i indexli sınıfın adı
        if i < len(class_names):
             # R-CNN sınıf indekslemesi bazen background'ı atlar veya dahil eder,
             # genellikle outputlarda background olmaz, class 1'den başlar.
             # O yüzden map_per_class[0] -> Genelde ilk gerçek sınıftır.
             # Eşleştirmeyi basit tutuyoruz:
             c_name = class_names[i+1] if (i+1) < len(class_names) else f"Class {i+1}"
             print(f"   - {c_name:10s}: {score.item():.4f}")
print("="*45)

📂 Model Yükleniyor: runs/detect/atik_faster_rcnn/best.pth
✅ Ağırlıklar başarıyla yüklendi.
📊 Faster R-CNN Karşılaştırma Testi Başlıyor...
⚠️ test_loader bulunamadı, val_loader kullanılıyor.


Faster R-CNN Test: 100%|██████████| 10/10 [00:13<00:00,  1.39s/it]



📊 FASTER R-CNN vs SSD/YOLO KARŞILAŞTIRMA
⚡ HIZ (Performans):
   - Toplam Süre      : 13.93 saniye
   - İşlenen Resim    : 155
   - FPS (Kare/Saniye): 11.13
---------------------------------------------
🎯 BAŞARI (Accuracy):
   - mAP@50 (Genel Başarı)     : 0.7090
   - mAP@50-95 (COCO Standardı): 0.3511
---------------------------------------------
📦 Sınıf Bazlı mAP@50-95 Değerleri:
   - cam       : 0.3126
   - kagit     : 0.2942
   - metal     : 0.4481
   - pil       : 0.4176
   - plastik   : 0.2831


In [7]:
!zip -r contentfastercnn.zip /content/runs/


  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/atik_faster_rcnn/ (stored 0%)
  adding: content/runs/detect/atik_faster_rcnn/best.pth (deflated 7%)
  adding: content/runs/detect/atik_faster_rcnn/last.pth (deflated 7%)


In [8]:
!cp /content/contentfastercnn.zip /content/drive/MyDrive/
